# How big is it? Measure a planet's size from the dip in its star's light

[Open this notebook in Colab](https://colab.research.google.com/github/Rakiura-Inc/limits-of-habitability/blob/main/notebooks/03_find_the_dip.ipynb)

When a planet crosses in front of its star, the star dims a little for a few hours. How much it dims tells you how big the planet is next to its star. In this notebook you download a star's light from TESS, find the dip, and turn its depth into the planet's radius. In the next notebook you weigh the same planet, and the two together tell you what it is made of.

This is a new planet, not the one from notebook 1. None of the small habitable zone planets there has both a TESS light curve and published velocities you can fit, so here you practise on bigger, hotter planets that have both. The same two measurements, made on a small planet, are how the lab's 0.8 line would be tested.

Run the cells in order (Shift+Enter). Boxes headed **Write down** ask you questions. Answer each one in the cell underneath by replacing its `...`, then run that cell: the notebook keeps your answer. The last cell puts all your answers into a single table and saves it with your plots. That table is what you hand in on the step.

In [ ]:
# Setup. Run this first. In Colab it installs what the notebook needs.
# If Colab asks you to restart the session afterwards, restart it and run the notebook from the top.
import importlib.util

# habtrain: a small package written for these notebooks. It downloads the data, fits orbits and draws the plots.
if importlib.util.find_spec("habtrain") is None:
    %pip install -q "habtrain @ git+https://github.com/Rakiura-Inc/limits-of-habitability@main"

# rakiura: the optional helper that connects this notebook to your step in Rakiura.
if importlib.util.find_spec("rakiura") is None:
    %pip install -q "rakiura @ git+https://github.com/Rakiura-Inc/rakiura-py@main"

import numpy as np                 # arrays and arithmetic
import matplotlib.pyplot as plt    # plots
import habtrain as ht              # the notebooks' package, used as ht.data, ht.fit, ht.plot and so on

# Which notebook this is, so your hand-in records exactly what you ran.
NOTEBOOK = "notebooks/03_find_the_dip.ipynb"
MODE = ht.portal.connect(NOTEBOOK)

> **Connect to Rakiura (optional)**
>
> Connect, and Rakiura ticks off each part as you finish it, with your plots on the step. You still attach your answers table and press Hand in on the step yourself. Skip it and everything still works: you attach the saved files on the step.
>
> To connect, run the cell below, open the link it prints, check the code matches the one shown, and click Allow. In Colab you do this once per session. To skip the click next time, add your key as a Colab secret named `RAKIURA_KEY` (the key icon on the left; your key is in Rakiura under Settings > Notebook key).

In [ ]:
# Connect to Rakiura (optional). Open the link it prints, check the code matches, click Allow.
MODE = ht.portal.connect(NOTEBOOK, sign_in=True)

## What a transit tells you

A planet in front of its star blocks a disc of the star's light. The fraction of light blocked, the **transit depth**, is the planet's disc divided by the star's:

depth = (planet radius ÷ star radius)<sup>2</sup>, so planet radius = star radius × √depth

The Sun is 109 times Earth's radius, so a planet the size of Earth in front of a star like the Sun blocks about 1/109<sup>2</sup>, or 84 parts per million.

**TESS**, the Transiting Exoplanet Survey Satellite, watches each patch of sky for about 27 days at a time (a sector), and for chosen stars records the brightness every 2 minutes ([Ricker et al. 2015](https://arxiv.org/abs/1406.0151)). The planets in this notebook were all found by TESS. Each is named for its **TOI** number (TESS Object of Interest, a star TESS flagged for a possible planet), and the letter after it marks the planet: "b" is the first found around that star.

## Worked example: TOI-1181 b

A hot Jupiter: a giant planet like Jupiter on an orbit so close to its star that it is very hot. It transits every 2.1 days, and you will weigh it in notebook 4. This works through every step once; then you repeat them on your own planet.

In [ ]:
# The orbit from the survey's published table: the period, and the time of one transit.
orb = ht.data.orbit("TOI-1181 b")
hours = ht.data.transit_duration("TOI-1181 b")   # how long each transit lasts, in hours

# Download two sectors of TESS 2-minute data for the star, joined into one light curve.
lc = ht.lightcurve.download("TOI-1181", sectors=2)
print(f"{len(lc)} brightness measurements; the transit lasts {hours:.1f} hours and repeats every {orb['P']:.4f} days")

# The raw light curve: the star's slow changes, the gaps between orbits, and the dips.
fig = ht.plot.light_curve(lc, "TOI-1181: brightness over two TESS sectors")
plt.show()

The short dips every 2.1 days are the transits. The slow wander underneath is the star and the spacecraft, not the planet, so the next cell takes it out. The transits are hidden from that smoothing so it cannot eat into them. Then the light curve is folded: every transit laid on top of the others, by hours from its middle.

In [ ]:
# Take out the slow changes, leaving the transits alone.
flat = ht.lightcurve.flatten(lc, orb["P"], orb["T0"], hours)

# Fold: every point placed by its hours from the nearest mid-transit.
h, flux = ht.lightcurve.fold(flat, orb["P"], orb["T0"])

# Average the points in 15-minute bins, so the shape stands out from the scatter.
bins = ht.lightcurve.binned(h, flux)
fig = ht.plot.transit(h, flux, bins, hours, "TOI-1181 b: every transit, folded")
plt.show()

In [ ]:
# The depth: the usual brightness outside the transit minus the brightness in the middle of it.
outside = np.abs(h) > hours / 2 * 1.5     # well clear of the transit on both sides
middle = np.abs(h) < hours / 2 * 0.6      # the flat bottom of the dip
depth = np.median(flux[outside]) - np.median(flux[middle])

# From depth to radius: the square root gives the planet's size as a fraction of the star's.
star_radius = ht.data.star_radius("TOI-1181 b")                 # in solar radii
radius = np.sqrt(depth) * star_radius * ht.data.SUN_IN_EARTH_RADII  # in Earth radii
print(f"Depth {depth * 1e6:.0f} parts per million. Star {star_radius} solar radii. "
      f"Planet {radius:.1f} Earth radii, about {radius / 11.2:.1f} times Jupiter's radius.")

## Your planet

The map below shows every planet you can measure here and weigh in notebook 4. Each is numbered.
- **Across: its orbital period.** A shorter period means more transits in each sector.
- **Up: how bright its star looks to TESS.** A brighter star means less scatter in the light curve.
- **Point size: how many velocities the survey took,** which matters in notebook 4.

In [ ]:
# Every planet you can measure here and weigh in notebook 4, with what is known before measuring them.
# TOI-1181 b, the worked example, is left out.
planets = ht.data.planets()
planets = planets[planets.planet != "TOI-1181 b"].reset_index(drop=True)
fig = ht.plot.planet_map(planets)
plt.show()

Choose a planet from the menu. The map circles your choice. You will weigh the same planet in notebook 4, so pick one you want to spend time on. TOI-669 b and TOI-1451 b are good first choices: both stars are bright, both dips are clear, and both have a published radius and mass to check against. If you would rather not choose, the first option picks one at random.

In [ ]:
# A menu of the planets, each described. Choosing one redraws the map with your choice circled.
chooser = ht.choose.planet(planets, ht.plot.planet_map)

> **Write down: your planet**
>
> - Choose your planet in the menu above. If the menu does not appear, put its number from the map in `MY_PICK`.
>
> **Where to look:** the map and the menu above.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

In [ ]:
MY_PICK = ...   # only if the menu did not appear: your planet's number on the map

# Your planet: what you chose in the menu, else the number you typed, else a random one.
planet = ht.choose.resolve(chooser, MY_PICK, planets, "planet", "planet")
ht.record.save("my_planet", planet, "your planet")

# Its orbit and transit length, then two sectors of its light curve, cleaned and folded as in the worked example.
orb = ht.data.orbit(planet)
hours = ht.data.transit_duration(planet)
lc = ht.lightcurve.download(planet.split()[0], sectors=2)
flat = ht.lightcurve.flatten(lc, orb["P"], orb["T0"], hours)
h, flux = ht.lightcurve.fold(flat, orb["P"], orb["T0"])
bins = ht.lightcurve.binned(h, flux)
fig_dip = ht.plot.transit(h, flux, bins, hours, f"{planet}: every transit, folded")
plt.show()
print(f"{len(lc)} brightness measurements; transit {hours:.1f} hours, period {orb['P']:.4f} days")
print(f"Your star's radius: {ht.data.star_radius(planet)} solar radii")

**Predict first**

Neptune is 3.88 Earth radii. If your planet were Neptune's size, how deep would its dip be in front of your star, in parts per million? Use the formula at the top: depth = (planet radius ÷ star radius)<sup>2</sup>, with the Sun's radius 109 Earth radii.

In [ ]:
# Your prediction: the depth, in parts per million, for a Neptune-sized planet in front of your star.
MY_PREDICTION = ...   # a number of parts per million, or "unsure"
ht.record.predict("neptune_depth", MY_PREDICTION)

> **Write down: your planet's dip and radius**
>
> - How deep is the dip? Write the code that measures it, as in the worked example, in `depth` (a fraction, not parts per million).
> - How big is the planet? Write the code that turns the depth into Earth radii in `radius`.
> - In one line: is the dip clear above the scatter, and is your planet bigger or smaller than Neptune, going by your prediction? Put it in `HOW_CLEAR`.
>
> **Where to look:** the folded plot. The blue points are the 15-minute averages; the shaded band is the transit.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

<details><summary>Stuck? Open hint 1</summary>

Copy all three lines from the worked example, so `outside` and `middle` are rebuilt for your planet's data: `outside = np.abs(h) > hours / 2 * 1.5`, `middle = np.abs(h) < hours / 2 * 0.6`, then `np.median(flux[outside]) - np.median(flux[middle])`.

</details>

<details><summary>Stuck? Open hint 2</summary>

`np.sqrt(depth) * ht.data.star_radius(planet) * ht.data.SUN_IN_EARTH_RADII`

</details>

In [ ]:
# This only runs once you have made your prediction above.
ht.record.revealed("neptune_depth")

# Your code: the depth of your planet's dip, then its radius in Earth radii.
depth = ...
radius = ...
HOW_CLEAR = ...   # one line, in quotes

# Print your result in parts per million and Earth radii, and keep it for your hand-in.
if ht.record.filled(depth) and ht.record.filled(radius):
    print(f"Depth {depth * 1e6:.0f} parts per million. Radius {radius:.2f} Earth radii.")
    ht.record.save("depth", round(depth * 1e6), "the depth of the dip (parts per million)")
    ht.record.save("radius", round(radius, 2), "your planet's radius (Earth radii)")
ht.record.save("how_clear", HOW_CLEAR, "how clear the dip is, and bigger or smaller than Neptune")

**Worth knowing**

The middle of a star is brighter than its edge, so a planet crossing the middle blocks a little more than its share of light. Turning the depth straight into a radius, as you just did, therefore usually gives a radius that is a little too big. The exception is a planet that crosses near the star's edge: the dip is then V-shaped, and the radius comes out a little too small. Fitting the full shape of the dip takes that out, and that is part of the lab's TESS project. Write down your radius as it is: you will use it in notebook 4.

## Hand in

Run the cell below. It shows all your answers in one table. Check them, then hand in this step in Rakiura, either by pasting the table in or by attaching the saved files. To get the files in Colab, click the folder icon on the left, open `handin`, then `dip-measured`, and download them.

Once you have handed in, the step shows the answer key: the published radius of every planet on the map.

In [ ]:
# Your answers and your folded transit, saved as your hand-in.
ht.record.handin("dip-measured", ["my_planet", "depth", "radius", "how_clear"], figures={"dip": fig_dip})